# SecondLook | 10 AI Analyst agent

**Goal:** a manager asks a question in plain English ("Which persona has the most repayers we decline?") and Claude answers by writing SQL and calling SecondLook's own tools, showing its work.

**How it works** (`src/secondlook/agent.py`). The loop is written by hand, with no framework, so every step can be explained:

1. Send the question, the tool definitions and the conversation so far to Claude Haiku 5.5.
2. If Claude asks for tools, run each one in Python and send the results back.
3. Repeat until Claude gives a final answer, or stop after **6 tool calls**.

Each step's tool, input and result are recorded, with tokens and cost, so the app can show the work.

| Tool | What it does |
|---|---|
| `run_sql(query)` | One read-only `SELECT` on `analyst.db`, at most 200 rows back |
| `explain_applicant(id)` | The model's SHAP reasons for one demo applicant (`explain.py`) |
| `path_to_approval(id)` | The re-scored loan changes that would get them approved (`path.py`) |
| `fairness_summary(group)` | The fairness audit for gender, age band or credit file (notebook 08) |

The table schemas are written into the `run_sql` tool description, so Claude knows the columns, their meanings and their codes without guessing.

## 1. Setup

In [1]:
import json
import sqlite3
import sys
import warnings
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "secondlook").exists())
sys.path.insert(0, str(ROOT / "src"))

from secondlook import agent as A
from secondlook import analyst_db
from secondlook.analyst_eval import expected_answers, grade
from secondlook.config import PROCESSED_DIR

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 200)
print(f"Model: {A.MODEL}   effort: {A.EFFORT}   max tool calls: {A.MAX_TOOL_CALLS}   row cap: {A.MAX_ROWS}   agent version: {A.AGENT_VERSION}")

Model: claude-haiku-5-5   effort: medium   max tool calls: 6   row cap: 200   agent version: v2


## 2. The analyst database

`analyst.db` is built only from the app bundle (`src/secondlook/analyst_db.py`), so it can ship to the cloud; the 2.3 GB working database never leaves this machine. Four tables: one row per test applicant (with decision, persona and path), one per persona, one per fairness group, and one per bootstrapped gap.

In [2]:
counts = analyst_db.build()
print(f"{analyst_db.DB_PATH.relative_to(ROOT)}: {analyst_db.DB_PATH.stat().st_size / 1e6:.1f} MB")
con = sqlite3.connect(f"file:{analyst_db.DB_PATH.as_posix()}?mode=ro", uri=True)
pd.DataFrame({"rows": counts, "columns": {t: len(con.execute(f"PRAGMA table_info({t})").fetchall()) for t in counts}})

app\demo_data\analyst.db: 6.7 MB


,rows,columns
portfolio,61503,13
personas,5,15
fairness,9,11
fairness_gaps,12,7


In [3]:
pd.read_sql("SELECT decision, COUNT(*) AS applicants, AVG(defaulted) AS default_rate FROM portfolio GROUP BY decision", con)

,decision,applicants,default_rate
0,approve,54240,0.053153
1,decline,4652,0.333835
2,under_review,2611,0.202604


The decision counts and default rates match notebook 03 and the Model Card: 54,240 approved with a 5.3% default rate, 2,611 under review and 4,652 declined.

## 3. Making `run_sql` safe

The page is public, so the SQL tool must not be able to change or break anything, whatever Claude (or a user's prompt) asks for. Four layers:

1. **One `SELECT` only:** the text must start with `SELECT` or `WITH`, contain no `;` between statements, and no write or settings keyword.
2. **A read-only file:** SQLite opens the file with `mode=ro`, so even a write that slipped past layer 1 would fail.
3. **A 200-row cap:** big results are cut, with a note telling Claude to aggregate instead.
4. **A 5-second limit:** a runaway query (for example, joining the table to itself) is stopped.

Errors and refusals go back to Claude **as text**, so it can fix its own query instead of crashing.

In [4]:
attacks = {
    "DROP": "DROP TABLE portfolio",
    "UPDATE": "UPDATE portfolio SET pd = 0",
    "Two statements": "SELECT COUNT(*) FROM portfolio; DELETE FROM portfolio",
    "Write hidden in a CTE": "WITH x AS (SELECT 1) DELETE FROM portfolio",
    "10,000-row query": "SELECT * FROM portfolio LIMIT 10000",
    "Runaway query (3.8 billion row join)": "SELECT COUNT(*) FROM portfolio a, portfolio b",
    "Typo (error goes back to Claude)": "SELECT nope FROM portfolio",
}
rows = []
for name, q in attacks.items():
    out = A.run_sql(q)
    n_rows = out.count("\n") - 1 if not out.startswith(("Refused", "SQL error")) else 0
    rows.append({"test": name, "result": out.splitlines()[-1] if n_rows else out, "data rows returned": max(n_rows, 0)})
pd.DataFrame(rows)

,test,result,data rows returned
0,DROP,Refused: only SELECT queries are allowed (the database is read-only).,0
1,UPDATE,Refused: only SELECT queries are allowed (the database is read-only).,0
2,Two statements,Refused: only one statement is allowed (found ';' between statements).,0
3,Write hidden in a CTE,Refused: the query contains a statement that changes data or settings; only reading is allowed.,0
4,"10,000-row query","[Only the first 200 rows are shown. Use aggregates (COUNT, AVG, GROUP BY) or LIMIT instead of listing rows.]",201
5,Runaway query (3.8 billion row join),SQL error: the query took longer than 5 seconds and was stopped,0
6,Typo (error goes back to Claude),SQL error: no such column: nope,0


In [5]:
# Defence in depth: even if a write got past the checks, the file is opened read-only
try:
    con.execute("DELETE FROM portfolio")
except sqlite3.OperationalError as e:
    print("Direct write on the read-only connection:", e)
print("Rows still in portfolio:", con.execute("SELECT COUNT(*) FROM portfolio").fetchone()[0])

Direct write on the read-only connection: attempt to write a readonly database
Rows still in portfolio: 61503


**Every attack is refused safely:** `DROP`, `UPDATE`, two statements, and a delete hidden inside a `WITH` are refused before running. The 10,000-row request returns only 200 rows plus a note, the 3.8-billion-row join is stopped after 5 seconds, and a typo comes back as a readable error. The direct write test shows the second layer on its own: the read-only connection refuses it, and the table still has all 61,503 rows.

## 4. Ten test questions, answered first with pandas

Each question's expected answer is computed in `src/secondlook/analyst_eval.py` **straight from the bundle files** (`portfolio.parquet`, `personas.parquet`, `fairness.json` and `path.py`), not from `analyst.db` or any SQL. So the check is independent: a wrong query can't make itself look right.

In [6]:
qs = expected_answers()
pd.DataFrame([{"#": q["id"], "question": q["question"], "tool needed": q["tool"],
               "expected (pandas)": "; ".join(f"{c['what']}: {c['value']:,}" for c in q["checks"]) +
                                    (f"; must name: {', '.join(q['words'])}" if q["words"] else "")} for q in qs]).set_index("#")

,question,tool needed,expected (pandas)
#,,,
1,"What is the approval rate for thin-file applicants, those with no credit bureau record?",run_sql,approval rate of thin files (%): 82.7
2,"How many applicants are under review, and what share of them actually repaid?",run_sql,"under review (count): 2,611; repaid among under review (%): 79.7"
3,"Which persona has the most repayers among the applicants we don't approve, and how many repayers is that?",run_sql,"repayers in top persona (count): 2,127; must name: Newer job, clean record"
4,What is the default rate of the loans we approve?,run_sql,default rate of approved (%): 5.3
5,"Of all the applicants who actually repaid, what percentage did we not approve?",run_sql,repayers not approved (%): 9.2
6,"Which age band has the lowest approval rate, and what is it?",run_sql,lowest age-band approval rate (%): 79.2; must name: 20-29
7,What share of declined applicants (not counting the under-review band) have a path to approval?,run_sql,declined with a path (%): 31.3
8,What is the average loan amount for approved applicants compared with applicants we don't approve?,run_sql,"mean loan, approved: 608,201; mean loan, not approved: 519,934"
9,"How much more often are women who repaid approved than men who repaid, and is that gap statistically real?",fairness_summary,gap (points): 4.2; interval low: 3.6; interval high: 4.7


The questions cover what a manager would ask: approval and default rates, the under-review band, personas, good borrowers turned away, an age-band comparison, paths to approval, a fairness gap with its interval, and one applicant's path. Eight are meant for SQL and two need SecondLook's own tools (Claude chose the fairness tool for the age-band question too, which also works).

**Grading** (`grade`): an answer is correct only if **every** expected number appears in it (to within 0.15 points for percentages, 0.5% for average amounts, and exactly for counts) and it names the right persona or age band. After reading the first run's answers we added one more rule (Section 6): Q9 fails if it claims the experiment removed gender.

## 5. Run the agent: two runs of all ten questions

Claude's answers vary from run to run, so every question is asked twice. Results are cached by agent version, so re-running the notebook costs nothing; changing the prompt or tools means bumping `AGENT_VERSION` and re-running.

In [7]:
CACHE = PROCESSED_DIR / "agent_runs.json"
cache = json.loads(CACHE.read_text(encoding="utf-8")) if CACHE.exists() else {}
client, toolbox = None, A.Toolbox()
RUNS = 2
for run in range(1, RUNS + 1):
    for q in qs:
        key = f"{A.AGENT_VERSION}|{run}|{q['id']}"
        if key not in cache:                     # only calls Claude for questions not run before
            if client is None:
                from secondlook.letters import _client
                client = _client()
            cache[key] = A.run_agent(q["question"], client, toolbox).to_dict()
            CACHE.write_text(json.dumps(cache, indent=1, ensure_ascii=False), encoding="utf-8")
results = []
for run in range(1, RUNS + 1):
    for q in qs:
        r = cache[f"{A.AGENT_VERSION}|{run}|{q['id']}"]
        g = grade(r["answer"], q)
        results.append({"run": run, "#": q["id"], "correct": g["correct"], "missing": "; ".join(g["missing"]),
                        "tools": ", ".join(s["tool"] for s in r["steps"]), "tool calls": len(r["steps"]),
                        "errors fixed": sum(s["is_error"] for s in r["steps"]), "cost ($)": r["cost_usd"],
                        "seconds": r["seconds"], "stop": r["stop"]})
res = pd.DataFrame(results)
print(res.groupby("run")["correct"].agg(["sum", "count"]).rename(columns={"sum": "correct", "count": "questions"}).to_string())
print(f"\nMean cost per question: ${res['cost ($)'].mean():.4f}   max: ${res['cost ($)'].max():.4f}   "
      f"mean time: {res['seconds'].mean():.1f}s   max tool calls: {res['tool calls'].max()}")
res[res["run"] == 1].drop(columns="run").set_index("#")

     correct  questions
run                    
1         10         10
2         10         10

Mean cost per question: $0.0008   max: $0.0010   mean time: 3.0s   max tool calls: 1


,correct,missing,tools,tool calls,errors fixed,cost ($),seconds,stop
#,,,,,,,,
1,True,,run_sql,1,0,0.000671,2.4,answered
2,True,,run_sql,1,0,0.000659,2.3,answered
3,True,,run_sql,1,0,0.000739,2.9,answered
4,True,,run_sql,1,0,0.000641,2.2,answered
5,True,,run_sql,1,0,0.000688,2.5,answered
6,True,,fairness_summary,1,0,0.000945,3.1,answered
7,True,,run_sql,1,0,0.000708,2.6,answered
8,True,,run_sql,1,0,0.000661,2.3,answered
9,True,,fairness_summary,1,0,0.000971,3.9,answered


**20 of 20 correct (10 of 10 in each run)**, against the guide's bar of 9 of 10.

- **Claude Haiku 5.5 is enough.** The guide says to switch to Sonnet 5.5 only if Haiku fails; it didn't.
- **Cheap and quick:** about **$0.0008 per question** (under a tenth of a cent) and about 3 seconds. The 10-question session limit on the public page caps one visitor at about one cent.
- **One tool call per question:** every question was answered with a single well-formed query or tool call, with no errors to fix. The 6-call limit is a safety net that these questions never reached.

## 6. Reading the answers, not just the score

The grader checks numbers. It can't tell whether the words around them are true, so we read every answer. Three are shown with the exact tool calls behind them.

In [8]:
for qid in (3, 9, 10):
    r = cache[f"{A.AGENT_VERSION}|1|{qid}"]
    print(f"Q{qid}: {qs[qid - 1]['question']}")
    for s in r["steps"]:
        shown = s["input"].get("query") or json.dumps(s["input"])
        print(f"  tool: {s['tool']}  input: {shown}")
    print("  answer:\n    " + r["answer"].replace("\n", "\n    ") + "\n")

Q3: Which persona has the most repayers among the applicants we don't approve, and how many repayers is that?
  tool: run_sql  input: SELECT persona, COUNT(*) AS not_approved, SUM(CASE WHEN defaulted = 0 THEN 1 ELSE 0 END) AS repayers FROM portfolio WHERE decision IN ('under_review','decline') AND persona IS NOT NULL GROUP BY persona ORDER BY repayers DESC
  answer:
    The "Newer job, clean record" persona has the most repayers among applicants we don't approve: 2,127 of its 2,983 not-approved applicants repaid (about 71.3%).
    
    For comparison, the other personas' repayer counts are:
    - Thin file: 965
    - Big loan for the income: 770
    - Missed payments before: 746
    - Long job, many past loans: 573
    
    These counts cover applicants with a decision of under review or decline.

Q9: How much more often are women who repaid approved than men who repaid, and is that gap statistically real?
  tool: fairness_summary  input: {"group": "gender"}
  answer:
    Women who rep

**What reading the answers caught (agent version v1 → v2).** The first version also scored 20 of 20, but reading it showed two problems:

1. **A false claim.** One Q9 answer said the retrain experiment removed "gender and other sensitive features". Gender was never a model input; the experiment removed age, job tenure relative to age, family status and children. The tool's output didn't say so explicitly, so Claude filled the gap with a guess. **Fix:** the fairness tool now states that gender was never an input, and the grader now fails any Q9 answer that claims otherwise. Re-graded with that rule, the v1 run would have scored 19 of 20.
2. **A rounding slip.** The tool gave the gap as 0.0415 (rounded to 4 decimals), and Claude reported 4.1 points instead of the audit's 4.2. **Fix:** 5 decimals, and a note that 0.0415 means 4.15 points.

We also told the agent never to call payments "monthly" (the data doesn't say how often they are made), the same mistake we fixed in the letters in Phase 1.

In v2, every answer leads with the number, gives the counts behind it, says what it covers ("under review or decline"), and explains the result in plain words.

## 7. Save the answers for offline tests

`tests/test_agent.py` replays these saved answers: it recomputes every expected value with pandas and checks that at least 9 of 10 are still correct, with no API call. If the data or the bundle changes, the test fails until the evaluation is re-run.

In [9]:
fixture = {str(q["id"]): {k: cache[f"{A.AGENT_VERSION}|1|{q['id']}"][k] for k in ("answer", "steps", "cost_usd")} for q in qs}
path = ROOT / "tests" / "fixtures" / "analyst_runs.json"
path.write_text(json.dumps({"agent_version": A.AGENT_VERSION, "model": A.MODEL, "runs": fixture}, indent=1, ensure_ascii=False), encoding="utf-8")
print(f"Saved {path.relative_to(ROOT)}")

Saved tests\fixtures\analyst_runs.json


## 8. Summary and limitations

**Answer:** the AI Analyst answers all ten test questions correctly in both runs (20 of 20) on Claude Haiku 5.5, for about $0.0008 a question. Its SQL tool refuses every write, multi-statement, oversized and runaway query, on a database file that is read-only anyway. Reading the answers found a false claim and a rounding slip that the number-only grader missed. Both are fixed, and the grader now checks for the claim.

**Limitations**
- **Ten questions are a small test.** They were written by us, about data we know. Harder or vaguer questions ("why do young people get declined?") will be less reliable, and the page shows every query so a manager can check.
- **The grader checks numbers and a few words**, not reasoning. Reading the answers remains part of the evaluation.
- **Answers vary between runs.** Two runs agreeing is reassuring, not proof.
- **The tools only see the test portfolio and the 300 demo applicants**, not live applications.
- **The prompt is part of the system.** Any change to it needs a new agent version and a fresh run of this notebook.